# 作业四：机器翻译计算BLEU
姓名：韩岳成
学号：524531910029

In [6]:
!pip install sentencepiece

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Inwt\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 加载模型和Tokenizer

本次作业将使用从huggingface中下载德英翻译模型`Helsinki-NLP/opus-mt-en-de`进行翻译测试，无需大家自行训练模型

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")


C:\Users\Inwt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


### Tokenizer

> 下面是一个例子，展示Tokenizer和模型的使用。理解下面的例子可能对你的作业有帮助。

```python

Tokenizer会将句子分割成一个个token，然后将每个token转化为一个数字，这个数字就是这个token在词表中的id。

In [8]:
# Sample English sentence
sentence = "Good morning! How are you?"
inputs = tokenizer.encode(sentence, return_tensors="pt")
print(inputs)

tensor([[2628, 2973,   68,  650,   48,   41,   31,    0]])


可以将token id映射到对应的分词token

In [9]:
tokens = tokenizer.convert_ids_to_tokens(inputs[0])
print(tokens)

['▁Good', '▁morning', '!', '▁How', '▁are', '▁you', '?', '</s>']


可以使用`decode`方法将token id转化回原来的句子

In [10]:
decoded_string = tokenizer.decode(inputs[0])
print(decoded_string)

Good morning! How are you?</s>


## 计算BLEU score

BLEU 是机器翻译评估中最常用的指标之一。

核心思想：

- 一个好的翻译结果，应该在 n-gram 上尽可能和参考翻译重叠。
- 为了避免模型生成过短的翻译，BLEU 引入了 简短惩罚（Brevity Penalty, BP）。

计算公式为：
$$
BLEU = BP \cdot \exp(\sum^N_{n=1} w_n \log p_n)
$$
其中：

- $p_n$：n-gram 的修正精确率（modified precision）
- $w_n$：权重（通常取均匀分布，$w_n = 1/N$）
- BP：简短惩罚，定义如下
$$
B P= \begin{cases}1 & \text { if } c>r \\ \exp \left(1-\frac{r}{c}\right) & \text { if } c \leq r\end{cases}
$$

- $c$：候选翻译的长度
- $r$：参考翻译的长度（取最接近候选翻译长度的参考）

### n-gram提取

In [11]:
from collections import Counter

def ngram_counts(tokens, n):
    """
    从一个分词后的句子中提取所有 n-gram，并统计频数。
    
    参数:
        tokens (list): 句子的分词结果
        n (int): n-gram 的阶数
    
    返回:
        Counter: n-gram -> 出现次数
    """
    ## TODO：统计句子中 n-gram 的频次（2分）
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return Counter(ngrams)


### 修正精确率（modified precision）

In [12]:
def modified_precision(candidate, references, n):
    """
    计算 n-gram 的修正精确率。
    
    参数:
        candidate (list): 候选翻译（分词后的列表）
        references (list of list): 多个参考翻译（每个参考为一个分词列表）
        n (int): n-gram 阶数
    
    返回:
        (clipped_count, total_count): clip 后的匹配数量与候选总数
    """
    cand_ngrams = ngram_counts(candidate, n)
    max_ref_counts = Counter()

    # TODO：计算所有参考翻译中的最大 n-gram 计数（2分）
    for ref in references:
        ref_ngrams = ngram_counts(ref, n)
        for ng in ref_ngrams:
            max_ref_counts[ng] = max(max_ref_counts[ng], ref_ngrams[ng])

    # 对候选中的 n-gram 进行截断计数
    clipped = {ng: min(count, max_ref_counts[ng]) for ng, count in cand_ngrams.items()}
    
    # TODO：返回匹配与候选的总数（1分）
    return sum(clipped.values()), sum(cand_ngrams.values())

### 简短惩罚（Brevity Penalty）

In [13]:
import math

def brevity_penalty(candidate, references):
    """
    计算简短惩罚 (BP)。
    
    参数:
        candidate (list): 候选翻译
        references (list of list): 参考翻译列表
    
    返回:
        float: BP 值
    """
    c = len(candidate)
    ref_lens = [len(ref) for ref in references]
    # TODO：选择与候选长度最接近的参考长度并计算BP值（3分）
    min_len = min(ref_lens, key=lambda ref_len: (abs(ref_len - c), ref_len))
    BP = 1 if c > min_len else math.exp(1 - min_len / c) if c > 0 else 0
    return BP


### BLEU分数

In [14]:
def bleu_score(candidate, references, max_n=4):
    """
    计算 BLEU 分数（默认 BLEU-4）。
    
    参数:
        candidate (list): 候选翻译
        references (list of list): 参考翻译
        max_n (int): 最大 n-gram 阶数（默认为 4）
    
    返回:
        float: BLEU 分数
    """
    # TODO：根据公式和以上函数计算BLEU分数（6分）
    precisions = []
    for n in range(1, max_n + 1):
        clipped_count, total_count = modified_precision(candidate, references, n)
        precisions.append(clipped_count / total_count if total_count > 0 else 0)
    BP = brevity_penalty(candidate, references)
    if min(precisions) > 0:
        score = BP * math.exp(sum(math.log(p) for p in precisions) / max_n)
    else:
        score = 0
    return score


## 测试与总结（6分）

### Sample 1

In [15]:
sentence = "The weather is nice today."
references = [
    ["Das", "Wetter", "ist", "heute", "schön", "."],
    ["Heute", "ist", "das", "Wetter", "schön", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=40, num_beams=4, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence)

print("Sample-1 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-1 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-1 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


Translated Sentence: Das Wetter ist heute schön.
Sample-1 BLEU-1: 0.6549846024623855
Sample-1 BLEU-2: 0.6341861143397762
Sample-1 BLEU-4: 0.5475182535069453


### Sample 2

In [16]:
sentence = "The conference will take place in Berlin next week, and many researchers from different countries are expected to attend."
references = [
    ["Die", "Konferenz", "findet", "nächste", "Woche", "in", "Berlin", "statt", ",", "und", "es", "wird", "erwartet", ",", "dass", "viele", "Forscher", "aus", "verschiedenen", "Ländern", "teilnehmen", "."],
    ["Nächste", "Woche", "wird", "die", "Konferenz", "in", "Berlin", "abgehalten", ",", "wobei", "zahlreiche", "Wissenschaftler", "aus", "unterschiedlichen", "Ländern", "erwartet", "werden", "."],
    ["Die", "Konferenz", "wird", "nächste", "Woche", "in", "Berlin", "stattfinden", ",", "und", "viele", "Forscher", "aus", "anderen", "Ländern", "sollen", "daran", "teilnehmen", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=64, num_beams=4, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence.split())

print("Sample-2 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-2 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-2 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


Translated Sentence: ['Die', 'Konferenz', 'findet', 'nächste', 'Woche', 'in', 'Berlin', 'statt,', 'an', 'der', 'viele', 'Forscher', 'aus', 'verschiedenen', 'Ländern', 'teilnehmen', 'werden.']
Sample-2 BLEU-1: 0.7210206394184336
Sample-2 BLEU-2: 0.683654745092498
Sample-2 BLEU-4: 0.5941900332409864


### Sample 3

In [17]:
sentence = "Artificial intelligence is rapidly developing, and it is increasingly being applied in areas such as healthcare, education, and autonomous driving."
references = [
    ["Die", "künstliche", "Intelligenz", "entwickelt", "sich", "rasant", "und", "wird", "zunehmend", "in", "Bereichen", "wie", "Gesundheitswesen", ",", "Bildung", "und", "autonomes", "Fahren", "eingesetzt", "."],
    ["Künstliche", "Intelligenz", "macht", "schnelle", "Fortschritte", "und", "findet", "immer", "häufiger", "Anwendung", "im", "Gesundheitswesen", ",", "in", "der", "Bildung", "und", "im", "autonomen", "Fahren", "."],
    ["Künstliche", "Intelligenz", "entwickelt", "sich", "schnell", "und", "wird", "mehr", "in", "Medizin", ",", "Schulen", "und", "selbstfahrenden", "Autos", "benutzt", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=40, num_beams=5, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence)

print("Sample-3 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-3 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-3 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


Translated Sentence: Künstliche Intelligenz entwickelt sich rapide und wird zunehmend in Bereichen wie Gesundheitswesen, Bildung und autonomes Fahren eingesetzt.
Sample-3 BLEU-1: 0.8235294117647058
Sample-3 BLEU-2: 0.7524469885568253
Sample-3 BLEU-4: 0.573057404379869


### 实验总结

本实验主要利用了 Hugging Face 上的 `Helsinki-NLP/opus-mt-en-de` 预训练模型实现了英德机器翻译，并在代码上手动构建了机器翻译最广泛使用的客观评价指标——**BLEU (Bilingual Evaluation Understudy)** 算法，以此对翻译结果进行评分。

在实现 BLEU 的过程中，核心验证了以下三个主要机制：
1. **n-gram 特征匹配**：将生成的翻译片段划分为 1~4 阶组合，与参考答案（References）比对。它证明了机器翻译评估不应局限于单个词汇的对照，必须同时关注局部的连贯性。
2. **修正精确率 (Modified Precision)**：针对候选文本中可能出现的重复词作弊现象进行“截断限制”，也就是选取参考翻译中同一个 n-gram 出现的最大次数作为理论上限，能够极大程度避免系统通过短句重复刷分。
3. **简短惩罚 (Brevity Penalty, BP)**：BLEU 的计算本质上是基于 Precision（准确率）。为了避免模型只翻译有把握的一两个词来骗取 100% 准确度，引入了相对长度验证。只要生成文本比最相符的参考句子短，就会受到严格的指数级惩罚系数削弱得分。

**测试现象与规律分析**：
基于 3 个 Sample 在 BLEU-1、BLEU-2 和 BLEU-4 的不同测试可以清晰地观察到：
1. **高阶 $N$ 导致得分递减**：无论是翻译天气短句还是含有复合长句的专业话语，随着 $n$-gram 从 1 增加到 4，对语序完整匹配的严苛度呈指数级上升，BLEU 分数必然出现明显的递减。BLEU-1 主要衡量了词汇是否准确传达，而 BLEU-4 更多反映了语法表达的流利性。
2. **语言特性带来的多样性挑战**：对于较长的句子（尤其是 Sample 2 与 3），因为德语具有严格的从句动词后置和极其灵活的构词方式，即便模型翻译的内容完全正确、语义完全传达，如果语序和人工给定的几条 Reference 不太一致，或者是采取了替换词，其高阶的 BLEU 得分依然很容易受到惩罚。这表明了 BLEU 在少量人工参考基准下单句评估存在的局限性。

总体而言，通过本次实验的分词提取、模型推理流程，以及手工搭建基于 `modified_precision` 分部评估，系统且深入地理解了 BLEU 惩罚机制在宏观机器翻译质量评价中的基本运作原理和优缺点。